In [50]:
import numpy as np
import pandas as pd
import models
from datetime import datetime
from database import engine, sessionDB
from nba_api.stats.static import players
from nba_api.stats.endpoints import playerawards, commonplayerinfo

In [51]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [52]:
players= players.get_players()
#players

In [53]:
models.Base.metadata.create_all(bind=engine)
db = sessionDB()

player testing and structure allat stuff

In [54]:
#using lebron cus hes the goat
lebron = [p for p in players if p["full_name"].lower() == "lebron james"]
lebron = lebron[0] #de-listifying (dont do this in prod)
pid = lebron["id"] #2544
info_df = commonplayerinfo.CommonPlayerInfo(player_id=pid).get_data_frames()[0] 

needed_columns = [
    "DISPLAY_FIRST_LAST", 
    "BIRTHDATE", 
    "COUNTRY", 
    "HEIGHT", 
    "WEIGHT", 
    "DRAFT_YEAR", 
    "DRAFT_ROUND", 
    "DRAFT_NUMBER"
]

#nicknames and school i need to use an agent for

info_df = info_df[needed_columns]
info_df = info_df.rename({"DISPLAY_FIRST_LAST": "NAME"}, axis=1)

info_df

,NAME,BIRTHDATE,COUNTRY,HEIGHT,WEIGHT,DRAFT_YEAR,DRAFT_ROUND,DRAFT_NUMBER
0,LeBron James,1984-12-30T00:00:00,USA,6-9,250,2003,1,1


actual player info parsing

In [80]:
#height to cm
def to_CM(height: str):
    height = height.split("-")
    return round(int(height[0]) * 30.48 + int(height[1]) * 2.54)

to_CM("6-9")

206

In [ ]:
def getPlayer(pid: int):
    info_df = commonplayerinfo.CommonPlayerInfo(player_id=pid).get_data_frames()[0] 

    needed_columns = [
        "DISPLAY_FIRST_LAST", 
        "BIRTHDATE", 
        "COUNTRY", 
        "HEIGHT", 
        "WEIGHT", 
        "DRAFT_YEAR", 
        "DRAFT_ROUND", 
        "DRAFT_NUMBER"
    ]

    #nicknames and school i need to use an agent for

    info_df = info_df[needed_columns]
    info_df = info_df.rename({"DISPLAY_FIRST_LAST": "NAME"}, axis=1)

    return info_df

bron_id = 2544

info_df = getPlayer(bron_id)

pinfo = models.Player(
    player_id = bron_id, #dont hardcode this in prod
    name = info_df["NAME"].iloc[0],
    #nicknames = 
    country = info_df["COUNTRY"].iloc[0],
    #school =
    birthdate = datetime.fromisoformat(info_df["BIRTHDATE"].iloc[0]).date(),
    height = to_CM(info_df["HEIGHT"].iloc[0]),
    weight = int(info_df["WEIGHT"].iloc[0]),
    draft_year = int(info_df["DRAFT_YEAR"].iloc[0]),
    draft_round = int(info_df["DRAFT_ROUND"].iloc[0]),
    draft_pick = int(info_df["DRAFT_NUMBER"].iloc[0]),
)

#db.rollback()
#db.add(pinfo)
#db.commit()
#db.close()


NAME            str
BIRTHDATE       str
COUNTRY         str
HEIGHT          str
WEIGHT          str
DRAFT_YEAR      str
DRAFT_ROUND     str
DRAFT_NUMBER    str
dtype: object

award testing and structure allat stuff

In [ ]:
#using lebron again cus hes the goat
lebron = [p for p in players if p["full_name"].lower() == "lebron james"]
lebron = lebron[0] #de-listifying (dont do this in prod)
lebron_id = lebron["id"] #2544
lebron_awards = playerawards.PlayerAwards(player_id=lebron_id).get_data_frames()[0]

acronym_hmap = {
    "All-Defensive Team": "DEF",''
    "All-NBA": "NBA",
    "All-Rookie Team": "ROOK",
    "NBA All-Star": "AS",
    "NBA All-Star Most Valuable Player": "ASMVP",
    "NBA Champion": "CHAMP",
    "NBA Cup All-Tournament Team": "CUP",
    "NBA Cup Most Valuable Player": "CUPMVP",
    "NBA Finals Most Valuable Player": "FMVP",
    "NBA Most Valuable Player": "MVP",
    "NBA Player of the Month": "POTM",
    "NBA Player of the Week": "POTW",
    "NBA Rookie of the Month": "ROTM",
    "NBA Rookie of the Year": "ROTY",
}

all_nba_set = {"DEF", "NBA", "ROOK", "CUP"}

#gotta add stat champs + dpoy/dpom + cfmvp + cpoy + dunk/3pt contest + teammate oty + hustle player oty

clean_lebron_awards_df = lebron_awards[lebron_awards["DESCRIPTION"].isin(acronym_hmap)]
clean_lebron_awards_df = clean_lebron_awards_df.drop([
    "FIRST_NAME", 
    "LAST_NAME", 
    "CONFERENCE", 
    "TYPE", 
    "SUBTYPE1", 
    "SUBTYPE2", 
    "SUBTYPE3",
    "MONTH",
    "WEEK",
    "PERSON_ID",
    "TEAM"
], axis=1)

clean_lebron_awards_df["ALL_NBA_TEAM_NUMBER"] = clean_lebron_awards_df["ALL_NBA_TEAM_NUMBER"].fillna(0)
clean_lebron_awards_df = clean_lebron_awards_df.replace(r'^\s*$', 0, regex=True) 
#^ gets rid of empty strings

clean_lebron_awards_df["DESCRIPTION"] = clean_lebron_awards_df["DESCRIPTION"].map(acronym_hmap)
clean_lebron_awards_df["DESCRIPTION"] = clean_lebron_awards_df.apply(
    lambda r: f"{r['DESCRIPTION']}{r['ALL_NBA_TEAM_NUMBER']}" 
    if r['DESCRIPTION'] in all_nba_set else r['DESCRIPTION'], 
    axis=1
)

#clean_lebron_awards_df

actual award parsing

In [ ]:
def getAwards(pid: int):
    awards_df = playerawards.PlayerAwards(player_id=pid).get_data_frames()[0]

    acronym_hmap = {
        "All-Defensive Team": "DEF",''
        "All-NBA": "NBA",
        "All-Rookie Team": "ROOK",
        "NBA All-Star": "AS",
        "NBA All-Star Most Valuable Player": "ASMVP",
        "NBA Champion": "CHAMP",
        "NBA Cup All-Tournament Team": "CUP",
        "NBA Cup Most Valuable Player": "CUPMVP",
        "NBA Finals Most Valuable Player": "FMVP",
        "NBA Most Valuable Player": "MVP",
        "NBA Player of the Month": "POTM",
        "NBA Player of the Week": "POTW",
        "NBA Rookie of the Month": "ROTM",
        "NBA Rookie of the Year": "ROTY",
    }

    all_nba_set = {"DEF", "NBA", "ROOK", "CUP"}

    awards_df = awards_df[awards_df["DESCRIPTION"].isin(acronym_hmap)]
    awards_df = awards_df.drop([
        "FIRST_NAME", 
        "LAST_NAME", 
        "CONFERENCE", 
        "TYPE", 
        "SUBTYPE1", 
        "SUBTYPE2", 
        "SUBTYPE3",
        "MONTH",
        "WEEK",
        "PERSON_ID",
        "TEAM"
    ], axis=1)

    awards_df["ALL_NBA_TEAM_NUMBER"] = awards_df["ALL_NBA_TEAM_NUMBER"].fillna(0)
    awards_df = awards_df.replace(r'^\s*$', 0, regex=True) 
    #^ gets rid of empty strings

    awards_df["DESCRIPTION"] = awards_df["DESCRIPTION"].map(acronym_hmap)
    awards_df["DESCRIPTION"] = awards_df.apply(
        lambda r: f"{r['DESCRIPTION']}{r['ALL_NBA_TEAM_NUMBER']}" 
        if r['DESCRIPTION'] in all_nba_set else r['DESCRIPTION'], 
        axis=1
    )

    return awards_df.drop(["ALL_NBA_TEAM_NUMBER"], axis=1)

bron_id = 2544

awards = getAwards(bron_id)

award_list = [
    models.Award(
        player_id = bron_id, #dont hardcode this in prod
        season = award.SEASON,
        award_name = award.DESCRIPTION
    )
    for award in awards.itertuples()
]


db.rollback()
#db.add_all(award_list)
#db.commit()
#db.close()

[<models.Award object at 0x000002A32C67BE70>, <models.Award object at 0x000002A32C67BCB0>, <models.Award object at 0x000002A32C24E7B0>, <models.Award object at 0x000002A32C24FF50>, <models.Award object at 0x000002A32C24FCB0>, <models.Award object at 0x000002A32C03C210>, <models.Award object at 0x000002A32C03C280>, <models.Award object at 0x000002A32C03C2F0>, <models.Award object at 0x000002A32C03C360>, <models.Award object at 0x000002A32C03C3D0>, <models.Award object at 0x000002A32C03C440>, <models.Award object at 0x000002A32C03C4B0>, <models.Award object at 0x000002A32C03C520>, <models.Award object at 0x000002A32C03C590>, <models.Award object at 0x000002A32C03C600>, <models.Award object at 0x000002A32C03C670>, <models.Award object at 0x000002A32C03C6E0>, <models.Award object at 0x000002A32C03C750>, <models.Award object at 0x000002A32C03C7C0>, <models.Award object at 0x000002A32C03C830>, <models.Award object at 0x000002A32C03C8A0>, <models.Award object at 0x000002A32C03C910>, <models.A

C:\Users\tirth\AppData\Local\Temp\ipykernel_24768\395575442.py:65: SAWarning: Session's state has been changed on a non-active transaction - this state will be discarded.
  db.rollback()
